# BO Forge four-variable CLI campaign

This notebook demonstrates the v0.3 CLI workflow on a four-variable continuous campaign. It uses terminal commands through `python -m bo_forge` rather than the `CampaignSession` API directly.

## 1. Setup

The notebook calls the CLI with `subprocess.run([sys.executable, "-m", "bo_forge", ...])`. This is the notebook-safe equivalent of running `bo-forge ...` in a terminal.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import IFrame, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = Path("configs/04_simple_4d_maximise_logei.yaml")
SEED_LOG_PATH = Path("examples/04_simple_4d_maximise_logei_campaign_log.csv")
WORKING_LOG_PATH = Path("examples/04_simple_4d_maximise_logei_working_log.csv")
SUGGESTIONS_PATH = Path("examples/04_simple_4d_maximise_logei_latest_suggestions.csv")
REPORT_PATH = Path("reports/04_cli_4d_campaign_report.txt")
PROGRESS_PATH = Path("reports/04_cli_4d_progress.pdf")
DIAGNOSTICS_PATH = Path("reports/04_cli_4d_diagnostics.pdf")
TARGET_OBSERVED_ROWS = 15

CAMPAIGN_ARGS = [
    "--config",
    str(CONFIG_PATH),
    "--log",
    str(WORKING_LOG_PATH),
]


def display_pdf(path: Path, height: int = 560) -> None:
    display(IFrame(src=f"../{path.as_posix()}", width="100%", height=height))

Check the active environment, create the working CSV with `init-log`, then seed it with the initial observations used for this example campaign.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "bo_forge", "doctor"],
    cwd=PROJECT_ROOT,
    check=True,
)


In [ ]:
(PROJECT_ROOT / WORKING_LOG_PATH).unlink(missing_ok=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "bo_forge",
        "init-log",
        "--config",
        str(CONFIG_PATH),
        "--log",
        str(WORKING_LOG_PATH),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

seed_log = pd.read_csv(PROJECT_ROOT / SEED_LOG_PATH)
seed_log.to_csv(PROJECT_ROOT / WORKING_LOG_PATH, index=False)
pd.read_csv(PROJECT_ROOT / WORKING_LOG_PATH)

## 2. Synthetic objective

In [ ]:
def simulate_activity(row: pd.Series) -> float:
    x1 = float(row["precursor_ratio"])
    x2 = (float(row["annealing_temperature"]) - 300.0) / 500.0
    x3 = (float(row["electrolyte_concentration"]) - 0.1) / 1.9
    x4 = (float(row["reaction_time"]) - 10.0) / 110.0

    peak = np.exp(
        -0.5
        * (
            ((x1 - 0.62) / 0.16) ** 2
            + ((x2 - 0.55) / 0.20) ** 2
            + ((x3 - 0.35) / 0.18) ** 2
            + ((x4 - 0.60) / 0.22) ** 2
        )
    )
    interaction = 0.12 * np.sin(4.0 * x1 + 2.0 * x3) + 0.08 * x2 * x4
    return float(1.0 + peak + interaction)


## 3. Inspect campaign state with CLI commands

In [ ]:
subprocess.run(
    [sys.executable, "-m", "bo_forge", "validate", *CAMPAIGN_ARGS],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "bo_forge", "status", *CAMPAIGN_ARGS],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "bo_forge", "summary", *CAMPAIGN_ARGS],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "bo_forge", "next-action", *CAMPAIGN_ARGS],
    cwd=PROJECT_ROOT,
    check=True,
)


## 4. Suggest and record one experiment

In [ ]:
def observed_count() -> int:
    log = pd.read_csv(PROJECT_ROOT / WORKING_LOG_PATH)
    return int((log["status"] == "observed").sum())


def run_one_cli_experiment() -> pd.DataFrame:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "bo_forge",
            "suggest",
            *CAMPAIGN_ARGS,
            "--batch-size",
            "1",
            "--output",
            str(SUGGESTIONS_PATH),
            "--append",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
    suggestion = pd.read_csv(PROJECT_ROOT / SUGGESTIONS_PATH)
    row = suggestion.iloc[0]
    activity = simulate_activity(row)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "bo_forge",
            "mark-observed",
            *CAMPAIGN_ARGS,
            "--row-id",
            str(row["row_id"]),
            "--objective-value",
            f"{activity:.6f}",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    log = pd.read_csv(PROJECT_ROOT / WORKING_LOG_PATH)
    return log.loc[log["row_id"] == row["row_id"]].copy()

In [ ]:
first_result = run_one_cli_experiment()
display(first_result[[
    "row_id",
    "iteration",
    "source",
    "precursor_ratio",
    "annealing_temperature",
    "electrolyte_concentration",
    "reaction_time",
    "activity",
]])


## 5. Continue the sequential CLI loop

The seed log starts with four observations. These additional runs complete the Sobol initial design and then let the same `suggest` command move into LogEI once enough observations exist.

In [ ]:
additional_results = []
while observed_count() < TARGET_OBSERVED_ROWS:
    additional_results.append(run_one_cli_experiment())

assert observed_count() == TARGET_OBSERVED_ROWS

recorded = pd.concat(additional_results, ignore_index=True)
display(recorded[[
    "row_id",
    "iteration",
    "source",
    "precursor_ratio",
    "annealing_temperature",
    "electrolyte_concentration",
    "reaction_time",
    "activity",
]])
subprocess.run(
    [sys.executable, "-m", "bo_forge", "summary", *CAMPAIGN_ARGS],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "bo_forge", "next-action", *CAMPAIGN_ARGS],
    cwd=PROJECT_ROOT,
    check=True,
)

## 6. Export report and diagnostics from the CLI

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "bo_forge",
        "report",
        *CAMPAIGN_ARGS,
        "--output",
        str(REPORT_PATH),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
print((PROJECT_ROOT / REPORT_PATH).read_text(encoding="utf-8"))


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "bo_forge",
        "plot",
        *CAMPAIGN_ARGS,
        "--kind",
        "progress",
        "--output",
        str(PROGRESS_PATH),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
display_pdf(PROGRESS_PATH)

subprocess.run(
    [
        sys.executable,
        "-m",
        "bo_forge",
        "plot",
        *CAMPAIGN_ARGS,
        "--kind",
        "diagnostics",
        "--output",
        str(DIAGNOSTICS_PATH),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
display_pdf(DIAGNOSTICS_PATH)
